# DKT: Hidden State Dimensionality vs. Predictive Performance

**Goal**: Use Deep Knowledge Tracing (DKT) to test the effective dimensionality of student knowledge.  
Unlike M-IRT (which assigns a static embedding per user), DKT models the **sequence** of interactions —  
capturing learning dynamics that a static model cannot.

**Key question**: How large does the LSTM hidden state need to be before AUC on next-step prediction saturates?  
The elbow point is an empirical estimate of the effective number of latent knowledge dimensions.

**Why DKT instead of M-IRT here**:
- M-IRT (K=1–16) gave flat AUC ≈ 0.824 — no improvement beyond 1 dimension
- M-IRT treats each interaction independently; DKT conditions on the full interaction history
- The data is naturally sequential (answer_number gives the round order)
- DKT handles sparsity natively: no user×item matrix needed, just ordered sequences

**Model**: Standard LSTM-DKT (Piech et al., 2015)  
- Input at step t: one-hot of (challenge_id × correct) → size 2 × N_CHALLENGES  
- LSTM hidden state: h_dim  
- Output: sigmoid over N_CHALLENGES → P(correct | challenge, history)  
- Loss: BCE on the actually-attempted challenge at each step

**Sweep**: h_dim ∈ {16, 32, 64, 128, 256}

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Hyperparameters
HIDDEN_DIMS   = [16, 32, 64, 128, 256]
N_EPOCHS      = 5
BATCH_SIZE    = 512
LR            = 1e-3
MIN_SEQ_LEN   = 5          # minimum interactions to include a user
TEST_FRAC     = 0.2        # fraction of each sequence held out for evaluation
N_USERS_SAMPLE = 20_000    # subsample for speed (full 100k is too slow on CPU)

## 1. Load and prepare sequences

In [ ]:
df_raw = pd.read_csv('../data/pix_data.csv', index_col=0)
df = df_raw[df_raw['answer_result'].isin(['ok', 'ko'])].copy()
df['correct'] = (df['answer_result'] == 'ok').astype(np.int8)

# Keep only challenges seen >= 50 times (same filter as M-IRT)
MIN_OBS = 50
valid_challenges = df['challenge_id'].value_counts()
valid_challenges = valid_challenges[valid_challenges >= MIN_OBS].index
df = df[df['challenge_id'].isin(valid_challenges)].copy()

# Encode challenge IDs
challenge_enc = LabelEncoder().fit(df['challenge_id'])
df['cid'] = challenge_enc.transform(df['challenge_id'])
N_CHALLENGES = df['cid'].nunique()
INPUT_DIM = 2 * N_CHALLENGES   # one-hot: (challenge × correct)

print(f'N_CHALLENGES = {N_CHALLENGES}  |  INPUT_DIM = {INPUT_DIM}')

# Sort each user's interactions by answer_number (round), then row order
df = df.sort_values(['user_id', 'answer_number']).reset_index(drop=True)

# Build per-user sequences of (cid, correct)
sequences = []
for uid, grp in df.groupby('user_id', sort=False):
    cids    = grp['cid'].values.astype(np.int32)
    corrects = grp['correct'].values.astype(np.int8)
    comps    = grp['competence_code'].values
    if len(cids) >= MIN_SEQ_LEN:
        sequences.append((cids, corrects, comps))

print(f'Users with >= {MIN_SEQ_LEN} interactions: {len(sequences):,}')
seq_lens = [len(s[0]) for s in sequences]
print(f'Sequence length: mean={np.mean(seq_lens):.1f}  median={np.median(seq_lens):.0f}  max={max(seq_lens)}')

In [ ]:
# Split each sequence: train = first (1-TEST_FRAC), test = last TEST_FRAC
# Predict step t+1 given steps 0..t (next-step prediction)

class DKTDataset(Dataset):
    """
    Each item is a (input_seq, target_cid, target_y) tuple for one user.
    - input_seq: (L,) encoded as cid * 2 + correct  → used to build one-hot
    - target_cid: (L,) challenge id at the NEXT step
    - target_y:   (L,) correct at the NEXT step
    Only the NEXT step's challenge is evaluated (sparse teacher forcing).
    """
    def __init__(self, seqs, split='train', test_frac=TEST_FRAC):
        self.items = []
        for cids, corrects, _ in seqs:
            L = len(cids)
            split_idx = max(1, int(L * (1 - test_frac)))
            if split == 'train':
                c = cids[:split_idx]
                y = corrects[:split_idx]
            else:
                c = cids
                y = corrects
            # input at t = (cid[t], correct[t]); target at t = (cid[t+1], correct[t+1])
            inp = c[:-1].astype(np.int32) * 2 + y[:-1].astype(np.int32)  # encode as cid*2+correct
            tgt_c = c[1:].astype(np.int32)
            tgt_y = y[1:].astype(np.float32)
            if split == 'test':
                # evaluate only on the held-out suffix
                inp_test   = inp[split_idx-1:]    # inputs from split point onward
                tgt_c_test = tgt_c[split_idx-1:]
                tgt_y_test = tgt_y[split_idx-1:]
                if len(inp_test) < 1:
                    continue
                # still feed full history as context
                self.items.append((inp, tgt_c, tgt_y, split_idx - 1))
            else:
                if len(inp) < 1:
                    continue
                self.items.append((inp, tgt_c, tgt_y, len(inp)))  # eval mask = full

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def collate_fn(batch):
    """Pad sequences in a batch; return packed representation."""
    # Sort by length descending (required for pack_padded_sequence)
    batch = sorted(batch, key=lambda x: len(x[0]), reverse=True)
    inps, tgt_cs, tgt_ys, eval_starts = zip(*batch)
    lengths = [len(x) for x in inps]
    max_len = lengths[0]

    inp_pad   = np.zeros((len(batch), max_len), dtype=np.int32)
    tgt_c_pad = np.zeros((len(batch), max_len), dtype=np.int32)
    tgt_y_pad = np.full((len(batch), max_len), -1.0, dtype=np.float32)  # -1 = masked

    for i, (inp, tgt_c, tgt_y, es) in enumerate(zip(inps, tgt_cs, tgt_ys, eval_starts)):
        L = len(inp)
        inp_pad[i, :L]   = inp
        tgt_c_pad[i, :L] = tgt_c
        tgt_y_pad[i, :L] = tgt_y

    return (
        torch.tensor(inp_pad,   dtype=torch.long),
        torch.tensor(tgt_c_pad, dtype=torch.long),
        torch.tensor(tgt_y_pad, dtype=torch.float32),
        torch.tensor(lengths,   dtype=torch.long),
        torch.tensor(eval_starts, dtype=torch.long),
    )


train_ds = DKTDataset(sequences, split='train')
test_ds  = DKTDataset(sequences, split='test')
print(f'Train sequences: {len(train_ds):,}  |  Test sequences: {len(test_ds):,}')

## 2. DKT model

In [ ]:
class DKT(nn.Module):
    """
    Standard LSTM-DKT (Piech et al., 2015).
    - Input: one-hot over (challenge × correct), dim = 2 * N_CHALLENGES
    - LSTM: single layer with hidden_dim units
    - Output: linear → sigmoid over N_CHALLENGES  (P(correct | challenge))
    """
    def __init__(self, n_challenges: int, hidden_dim: int, dropout: float = 0.0):
        super().__init__()
        self.n_challenges = n_challenges
        self.hidden_dim   = hidden_dim
        input_dim = 2 * n_challenges
        self.lstm   = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc     = nn.Linear(hidden_dim, n_challenges)

    def forward(self, inp_idx, lengths):
        """
        inp_idx : (B, T) encoded as cid*2+correct
        lengths : (B,) actual sequence lengths
        Returns  : (B, T, N_CHALLENGES) logits
        """
        B, T = inp_idx.shape
        # Build one-hot on CPU, move to device
        x = torch.zeros(B, T, 2 * self.n_challenges, device=inp_idx.device)
        x.scatter_(2, inp_idx.unsqueeze(-1), 1.0)

        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        out_packed, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out_packed, batch_first=True)  # (B, T, H)
        out = self.dropout(out)
        logits = self.fc(out)  # (B, T, N_CHALLENGES)
        return logits

    def predict_correct(self, logits, tgt_c):
        """
        Extract the predicted probability for the target challenge at each step.
        logits : (B, T, N_CHALLENGES)
        tgt_c  : (B, T) challenge id at next step
        Returns: (B, T) probabilities
        """
        probs = torch.sigmoid(logits)                              # (B, T, N)
        tgt_probs = probs.gather(2, tgt_c.unsqueeze(-1)).squeeze(-1)  # (B, T)
        return tgt_probs

print('DKT model class defined.')
print(f'Input dim per step: 2 × {N_CHALLENGES} = {INPUT_DIM}')

## 3. Training and evaluation loop

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for inp, tgt_c, tgt_y, lengths, eval_starts in loader:
            inp    = inp.to(device)
            tgt_c  = tgt_c.to(device)
            tgt_y  = tgt_y  # keep on CPU for AUC
            logits = model(inp, lengths)
            probs  = model.predict_correct(logits, tgt_c).cpu()  # (B, T)

            B, T = probs.shape
            for i in range(B):
                es = eval_starts[i].item()
                L  = lengths[i].item()
                # evaluate only on the test suffix
                for t in range(es, L):
                    if tgt_y[i, t].item() >= 0:   # not masked
                        all_probs.append(probs[i, t].item())
                        all_labels.append(tgt_y[i, t].item())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    if len(np.unique(all_labels)) < 2:
        return float('nan'), float('nan')
    bce  = -np.mean(all_labels * np.log(all_probs + 1e-9) + (1 - all_labels) * np.log(1 - all_probs + 1e-9))
    auc  = roc_auc_score(all_labels, all_probs)
    return bce, auc


def train_dkt(hidden_dim, train_ds, test_ds, n_epochs=N_EPOCHS, lr=LR, verbose=True):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=collate_fn, num_workers=0)

    model = DKT(N_CHALLENGES, hidden_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(reduction='none')

    history = {'train_loss': [], 'test_bce': [], 'test_auc': []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss, n_steps = 0.0, 0
        for inp, tgt_c, tgt_y, lengths, _ in train_loader:
            inp   = inp.to(DEVICE)
            tgt_c = tgt_c.to(DEVICE)
            tgt_y = tgt_y.to(DEVICE)

            logits = model(inp, lengths)                             # (B, T, N)
            # Extract logit for the target challenge
            tgt_logit = logits.gather(2, tgt_c.unsqueeze(-1)).squeeze(-1)  # (B, T)

            mask = (tgt_y >= 0)                                      # ignore padding
            loss = criterion(tgt_logit[mask], tgt_y[mask])
            loss = loss.mean()

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * mask.sum().item()
            n_steps    += mask.sum().item()

        train_loss = total_loss / max(n_steps, 1)
        test_bce, test_auc = evaluate(model, test_loader, DEVICE)

        history['train_loss'].append(train_loss)
        history['test_bce'].append(test_bce)
        history['test_auc'].append(test_auc)

        if verbose:
            print(f'  h={hidden_dim:3d} | epoch {epoch:2d} | '
                  f'train_loss={train_loss:.4f} | '
                  f'test_bce={test_bce:.4f} | '
                  f'test_auc={test_auc:.4f}')

    return model, history

print('Training function defined.')

## 4. Sweep over hidden dimensions

In [ ]:
results = {}

for h in HIDDEN_DIMS:
    n_params = DKT(N_CHALLENGES, h).state_dict()
    n_params = sum(p.numel() for p in DKT(N_CHALLENGES, h).parameters())
    print(f'\n=== DKT hidden_dim={h}  |  params={n_params:,} ===')
    model, history = train_dkt(h, train_ds, test_ds, verbose=True)
    results[h] = {'model': model, 'history': history, 'n_params': n_params}

print('\nDone.')

## 5. Learning curves

In [ ]:
colors = cm.viridis(np.linspace(0, 1, len(HIDDEN_DIMS)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for h, color in zip(HIDDEN_DIMS, colors):
    hist = results[h]['history']
    axes[0].plot(hist['test_bce'], label=f'h={h}', color=color, lw=2)
    axes[1].plot(hist['test_auc'], label=f'h={h}', color=color, lw=2)

for ax, ylabel, title in zip(axes,
                              ['Test BCE (↓)', 'Test AUC-ROC (↑)'],
                              ['Test BCE loss', 'Test AUC-ROC']):
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('DKT: learning curves by hidden dimension', fontsize=13)
plt.tight_layout()
plt.savefig('../results/dkt_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Elbow plot + comparison with M-IRT

In [ ]:
best_aucs  = {h: max(results[h]['history']['test_auc'])  for h in HIDDEN_DIMS}
final_aucs = {h: results[h]['history']['test_auc'][-1]   for h in HIDDEN_DIMS}
best_bces  = {h: min(results[h]['history']['test_bce'])  for h in HIDDEN_DIMS}

summary_rows = []
for h in HIDDEN_DIMS:
    summary_rows.append({
        'hidden_dim': h,
        'n_params': results[h]['n_params'],
        'best_test_auc':  best_aucs[h],
        'final_test_auc': final_aucs[h],
        'best_test_bce':  best_bces[h],
    })
summary = pd.DataFrame(summary_rows).set_index('hidden_dim')
summary['delta_auc_vs_h16'] = summary['final_test_auc'] - summary.loc[16, 'final_test_auc']
print(summary.to_string(float_format='%.5f'))

# M-IRT best AUC for reference
MIRT_AUC = 0.82379  # K=1 final AUC from mirt_dimensionality.ipynb

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(HIDDEN_DIMS, [best_aucs[h] for h in HIDDEN_DIMS],
             'o-', color='steelblue', lw=2, markersize=8, label='DKT best AUC')
axes[0].axhline(MIRT_AUC, color='tomato', ls='--', lw=2, label=f'M-IRT K=1 AUC={MIRT_AUC:.4f}')
axes[0].set_xlabel('Hidden dimension')
axes[0].set_ylabel('Best test AUC-ROC')
axes[0].set_title('DKT: AUC vs hidden dimension (elbow plot)')
axes[0].set_xticks(HIDDEN_DIMS)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(HIDDEN_DIMS, [best_bces[h] for h in HIDDEN_DIMS],
             'o-', color='steelblue', lw=2, markersize=8)
axes[1].set_xlabel('Hidden dimension')
axes[1].set_ylabel('Best test BCE (↓)')
axes[1].set_title('DKT: BCE loss vs hidden dimension')
axes[1].set_xticks(HIDDEN_DIMS)
axes[1].grid(True, alpha=0.3)

plt.suptitle('DKT dimensionality sweep', fontsize=13)
plt.tight_layout()
plt.savefig('../results/dkt_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Per-competence AUC (best model)

Which competences benefit most from the sequential model? Evaluate predictions broken down by the competence of the *next* challenge.

In [ ]:
# Map challenge_id → competence_code
cid_to_comp = (
    df.drop_duplicates('cid')[['cid', 'competence_code']]
    .set_index('cid')['competence_code']
    .to_dict()
)

# Pick best-AUC model
best_h = max(best_aucs, key=best_aucs.get)
print(f'Best model: hidden_dim={best_h} (AUC={best_aucs[best_h]:.4f})')

best_model = results[best_h]['model']
best_model.eval()
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=0)

# Collect per-competence predictions
comp_probs  = {}
comp_labels = {}

with torch.no_grad():
    for inp, tgt_c, tgt_y, lengths, eval_starts in test_loader:
        inp_d  = inp.to(DEVICE)
        tgt_cd = tgt_c.to(DEVICE)
        logits = best_model(inp_d, lengths)
        probs  = best_model.predict_correct(logits, tgt_cd).cpu().numpy()  # (B, T)

        B, T = probs.shape
        for i in range(B):
            es = eval_starts[i].item()
            L  = lengths[i].item()
            for t in range(es, L):
                if tgt_y[i, t].item() < 0:
                    continue
                c = tgt_c[i, t].item()
                comp = cid_to_comp.get(c, None)
                if comp is None:
                    continue
                comp_probs.setdefault(comp, []).append(probs[i, t])
                comp_labels.setdefault(comp, []).append(tgt_y[i, t].item())

comp_aucs = {}
for comp in sorted(comp_probs.keys()):
    p = np.array(comp_probs[comp])
    y = np.array(comp_labels[comp])
    if len(np.unique(y)) >= 2:
        comp_aucs[comp] = roc_auc_score(y, p)

comp_auc_df = pd.Series(comp_aucs, name='AUC').sort_index()
print('\nPer-competence AUC (next-step prediction):')
print(comp_auc_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
comps = list(comp_auc_df.index)
aucs  = comp_auc_df.values
bars = ax.bar([str(c) for c in comps], aucs, color='steelblue', alpha=0.8)
ax.axhline(best_aucs[best_h], color='tomato', ls='--', lw=1.5, label=f'Overall AUC={best_aucs[best_h]:.4f}')
ax.axhline(MIRT_AUC, color='gray', ls=':', lw=1.5, label=f'M-IRT K=1 AUC={MIRT_AUC:.4f}')
ax.set_xlabel('Competence')
ax.set_ylabel('AUC-ROC')
ax.set_title(f'DKT (h={best_h}): per-competence next-step prediction AUC')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('../results/dkt_per_competence_auc.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary

In [ ]:
print('=== DKT Dimensionality Summary ===')
print(f'N_CHALLENGES={N_CHALLENGES}  INPUT_DIM={INPUT_DIM}')
print(f'M-IRT (K=1) reference AUC: {MIRT_AUC:.4f}\n')

for h in HIDDEN_DIMS:
    delta = best_aucs[h] - MIRT_AUC
    print(
        f'  h={h:3d}  params={results[h]["n_params"]:,}  '
        f'best_AUC={best_aucs[h]:.4f}  '
        f'Δ_vs_MIRT={delta:+.4f}'
    )

print(f'\nBest hidden dim: h={best_h}  AUC={best_aucs[best_h]:.4f}')
print(f'\nPer-competence AUC range: '
      f'{comp_auc_df.min():.4f} – {comp_auc_df.max():.4f}')
best_comp = comp_auc_df.idxmax()
worst_comp = comp_auc_df.idxmin()
print(f'  Best predicted: competence {best_comp}  (AUC={comp_auc_df[best_comp]:.4f})')
print(f'  Worst predicted: competence {worst_comp}  (AUC={comp_auc_df[worst_comp]:.4f})')